# Hugging Face image tasks — task-specific models
**6 tasks · 6 task-specific models · 2 examples per task**

Python 3.11.9 recommended. Models download from Hugging Face and run **locally**, using a token loaded from `.env`. There are no paid hosted inference calls in this notebook. Hardware, electricity, internet and any rented GPU can cost money. Model licenses still apply; free download does not mean unrestricted use.

These notebooks use task-specific Transformers/Diffusers models directly. Each example calls the model pipeline as a regular Python function, without LangChain wrappers.

Run setup, then each task's loading cell and both example cells. Cleanup releases RAM. CPU works for Tasks 1–5. Task 6 (diffusion) is much larger: a GPU with roughly 8 GB VRAM is recommended; CPU generation can take many minutes and needs substantial RAM. Running everything downloads several GB.

Validation: JSON, cell syntax and example counts checked. Full model downloads and inference were not executed during creation.

## Install
Use a clean environment. Run once and restart the kernel. Transformers v4 is intentional. This is a constrained environment, not a fully tested lockfile.

In [ ]:
%pip install -q "torch>=2.6,<3" "torchvision>=0.21,<1" "transformers==4.57.6" "langchain-huggingface==0.3.1" "langchain-core>=0.3.70,<1" "huggingface-hub>=0.34,<1" "python-dotenv>=1,<2" "numpy>=1.26,<3" "Pillow>=10,<12" "scikit-image>=0.23,<1" "diffusers>=0.35,<0.36" "accelerate>=1,<2" "sentencepiece>=0.2,<0.3"

## Load token from `.env`
Create `.env` beside the notebook (or upload it to `/content` in Colab):
```dotenv
HUGGINGFACE_API_KEY=hf_replace_with_your_token
```
Use a Hugging Face read token from https://huggingface.co/settings/tokens. It authenticates model downloads; it is not an OpenAI API key.

In [ ]:
import os, gc
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path.cwd() / ".env")
HF_TOKEN = os.getenv("HUGGINGFACE_API_KEY") or os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not HF_TOKEN or HF_TOKEN == "hf_replace_with_your_token":
    raise ValueError("Add your token to .env, then rerun this cell.")
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
import torch
import numpy as np
from transformers import pipeline, set_seed
from IPython.display import display, Audio
set_seed(42)
DEVICE = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if DEVICE == 0 else "CPU")

## Two real sample images
Uses the bundled scikit-image photographs: an astronaut and a cat. No image uploads are needed. Replace these with `Image.open("your_image.jpg").convert("RGB")` if desired. See scikit-image's data documentation for sample provenance; model licensing and photo licensing are separate.

In [ ]:
from PIL import Image, ImageDraw
from skimage import data
image1 = Image.fromarray(data.astronaut()).convert("RGB")
image2 = Image.fromarray(data.chelsea()).convert("RGB")
display(image1.resize((300, 300)), image2.resize((300, 225)))

## Task 1: Image classification
**Model:** [google/vit-base-patch16-224](https://huggingface.co/google/vit-base-patch16-224)

Returns ImageNet class labels; they may be more specific than person or cat.

In [ ]:
pipe = pipeline('image-classification', model='google/vit-base-patch16-224', token=HF_TOKEN, device=DEVICE)


### Example 1

In [ ]:
display(image1)
result = pipe(image1, top_k=3)
print(result)

### Example 2

In [ ]:
display(image2)
result = pipe(image2, top_k=3)
print(result)

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Task 2: Object detection
**Model:** [hustvl/yolos-tiny](https://huggingface.co/hustvl/yolos-tiny)

Returns object labels, confidence and bounding boxes.

In [ ]:
pipe = pipeline('object-detection', model='hustvl/yolos-tiny', token=HF_TOKEN, device=DEVICE)


### Example 1

In [ ]:
result = pipe(image1, threshold=0.5)
annotated = image1.copy()
draw = ImageDraw.Draw(annotated)
for item in result:
    box = item["box"]
    draw.rectangle([box["xmin"], box["ymin"], box["xmax"], box["ymax"]], outline="red", width=3)
    draw.text((box["xmin"], box["ymin"]), item["label"], fill="red")
print(result)
display(annotated)

### Example 2

In [ ]:
result = pipe(image2, threshold=0.5)
annotated = image2.copy()
draw = ImageDraw.Draw(annotated)
for item in result:
    box = item["box"]
    draw.rectangle([box["xmin"], box["ymin"], box["xmax"], box["ymax"]], outline="red", width=3)
    draw.text((box["xmin"], box["ymin"]), item["label"], fill="red")
print(result)
display(annotated)

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Task 3: Image captioning
**Model:** [Salesforce/blip-image-captioning-base](https://huggingface.co/Salesforce/blip-image-captioning-base)

Produces a short image description. Captions can be inaccurate.

In [ ]:
pipe = pipeline('image-to-text', model='Salesforce/blip-image-captioning-base', token=HF_TOKEN, device=DEVICE)


### Example 1

In [ ]:
display(image1)
result = pipe(image1, max_new_tokens=40)
print(result)

### Example 2

In [ ]:
display(image2)
result = pipe(image2, max_new_tokens=40)
print(result)

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Task 4: Visual question answering
**Model:** [dandelin/vilt-b32-finetuned-vqa](https://huggingface.co/dandelin/vilt-b32-finetuned-vqa)

Answers a short question from an image using a learned answer vocabulary.

In [ ]:
pipe = pipeline('visual-question-answering', model='dandelin/vilt-b32-finetuned-vqa', token=HF_TOKEN, device=DEVICE)


### Example 1

In [ ]:
display(image1)
result = pipe({"image": image1, "question": 'What is the person wearing?'}, top_k=3)
print(result)

### Example 2

In [ ]:
display(image2)
result = pipe({"image": image2, "question": 'What animal is shown?'}, top_k=3)
print(result)

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Task 5: Semantic segmentation
**Model:** [facebook/mask2former-swin-tiny-ade-semantic](https://huggingface.co/facebook/mask2former-swin-tiny-ade-semantic)

Groups pixels into ADE20K classes. The overlay shows up to five returned segments.

In [ ]:
pipe = pipeline('image-segmentation', model='facebook/mask2former-swin-tiny-ade-semantic', token=HF_TOKEN, device=DEVICE)


### Example 1

In [ ]:
segments = pipe(image1)
print("Segments:", [item["label"] for item in segments])
overlay = image1.convert("RGBA")
colors = [(255,0,0,90), (0,255,0,90), (0,0,255,90), (255,255,0,90), (255,0,255,90)]
for segment, color in zip(segments, colors):
    mask = segment["mask"].resize(image1.size).convert("L")
    layer = Image.new("RGBA", image1.size, color)
    layer.putalpha(mask.point(lambda p: int(p * 0.35)))
    overlay = Image.alpha_composite(overlay, layer)
display(overlay)

### Example 2

In [ ]:
segments = pipe(image2)
print("Segments:", [item["label"] for item in segments])
overlay = image2.convert("RGBA")
colors = [(255,0,0,90), (0,255,0,90), (0,0,255,90), (255,255,0,90), (255,0,255,90)]
for segment, color in zip(segments, colors):
    mask = segment["mask"].resize(image2.size).convert("L")
    layer = Image.new("RGBA", image2.size, color)
    layer.putalpha(mask.point(lambda p: int(p * 0.35)))
    overlay = Image.alpha_composite(overlay, layer)
display(overlay)

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Task 6: Text-to-image generation
**Model:** [CompVis/stable-diffusion-v1-4](https://huggingface.co/CompVis/stable-diffusion-v1-4)

CreativeML OpenRAIL-M license includes use restrictions. Downloads several GB. GPU recommended; 384×384 images and 20 steps keep the demo smaller. This is not a tiny model. The standard safety checker remains enabled.

In [ ]:
from diffusers import StableDiffusionPipeline
sd_device = "cuda" if torch.cuda.is_available() else "cpu"
sd_dtype = torch.float16 if sd_device == "cuda" else torch.float32
pipe = StableDiffusionPipeline.from_pretrained(
    "CompVis/stable-diffusion-v1-4", token=HF_TOKEN, torch_dtype=sd_dtype,
)
pipe = pipe.to(sd_device)
pipe.enable_attention_slicing()


### Example 1

In [ ]:
prompt = 'A watercolor painting of a peaceful lake surrounded by green mountains'
result = pipe(prompt, height=384, width=384, num_inference_steps=20,
                      generator=torch.Generator(device=sd_device).manual_seed(1))
display(result.images[0])

### Example 2

In [ ]:
prompt = 'A friendly small robot reading a book in a bright library, digital illustration'
result = pipe(prompt, height=384, width=384, num_inference_steps=20,
                      generator=torch.Generator(device=sd_device).manual_seed(2))
display(result.images[0])

Release the model from RAM. Rerun its loading cell to repeat this task.

In [ ]:
del chain, pipe
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Cost, access and troubleshooting
These models are publicly downloadable and the notebook makes no paid inference calls. Licenses differ; read the linked cards before redistribution or commercial deployment. Stable Diffusion has OpenRAIL use restrictions.

If imports fail, restart the kernel after installation. Install matching torch/torchvision builds for your platform. If a download needs an agreement, open the model page and complete it using the same account as your token. If GPU memory is exhausted, run cleanup or restart the kernel. CPU execution avoids GPU rental fees but is much slower for diffusion. Predictions and generated images are not guaranteed accurate.

[Transformers v4 pipelines](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines) · [Diffusers](https://huggingface.co/docs/diffusers/index)